# Phase 4 — Latency Harness: RSQR vs Continuous Re-rotation

Answers RFC §5#1: is batched, eviction-boundary-only rotation cheaper
in wall-clock terms than continuous, every-step rotation?

**Scope note:** dummy K/V tensors only, no real model. This isolates
rotation cost itself (kernel-launch overhead / call count), which is
what #1 is actually asking about -- precision was already settled in
Phase 3. Runs on Colab's T4 to sidestep local Pascal/CUDA version
issues; precision doesn't enter into this measurement at all, so the
T4 vs local GPU choice has zero bearing on result validity here.


In [ ]:
import torch
import time
import csv
import statistics
from dataclasses import dataclass, field

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("WARNING: no CUDA device found -- falling back to CPU. "
          "GPU timings are far more meaningful for this comparison "
          "(the whole question is about kernel-launch overhead, which "
          "barely exists on CPU). Check Runtime > Change runtime type > T4 GPU.")
else:
    print(f"Using device: {device} ({torch.cuda.get_device_name(0)})")


Using device: cuda (Tesla T4)


## RoPE implementation (from scratch, matches Phase 3's `apply_rope`)

Same math as the notebook's precision work -- `theta=1000000.0`
(Qwen2.5 default), fp32. Deliberately reimplemented here rather than
imported from a library, so both arms below share identical rotation
cost -- the comparison is about call pattern (batched vs continuous),
not implementation quality differences.


In [ ]:
ROPE_THETA = 1000000.0  # Qwen2.5 default -- matches Phase 3 precision runs

def rope_freqs(head_dim, theta=ROPE_THETA):
    assert head_dim % 2 == 0
    i = torch.arange(0, head_dim // 2, dtype=torch.float32, device=device)
    return theta ** (-2.0 * i / head_dim)

def apply_rope(x, positions, freqs):
    """x: [..., seq_len, head_dim]; positions: [seq_len]. Standard RoPE,
    identical math to the Phase 3 notebook's apply_rope."""
    if not isinstance(positions, torch.Tensor):
        positions = torch.tensor(positions, dtype=torch.float32, device=x.device)
    else:
        positions = positions.to(device=x.device, dtype=torch.float32)
    freqs = freqs.to(x.device)

    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]

    angles = positions[:, None] * freqs[None, :]
    cos = torch.cos(angles).to(x.dtype)
    sin = torch.sin(angles).to(x.dtype)
    while cos.dim() < x.dim():
        cos = cos.unsqueeze(0)
        sin = sin.unsqueeze(0)

    out1 = x1 * cos - x2 * sin
    out2 = x2 * cos + x1 * sin
    return torch.cat([out1, out2], dim=-1)


## Config and dummy tensor shapes

Dummy K/V tensors match a real KV cache's shape:
`[batch_size, n_kv_heads, seq_len, head_dim]`, fp32, `torch.randn` --
content doesn't matter since RoPE is pure elementwise math on shape.


In [ ]:
BATCH_SIZE = 1
N_KV_HEADS = 2
HEAD_DIM = 64

FREQS = rope_freqs(HEAD_DIM)

@dataclass
class BenchConfig:
    n_steps: int
    evict_every: int = 8       # matches Phase 3's block_size=8
    survivor_delta: int = 8    # matches Phase 3's survivor_delta=8
    n_trials: int = 20
    warmup_trials: int = 10


## Arm A -- continuous re-rotation (StreamingLLM-style, strict reading)

**Fixed from v1.** The first version of this cell gated Arm A's
rotation on the same `evict_every` cadence as Arm B, which made
`n_rotation_calls` identical between arms by construction -- the
benchmark was only measuring payload-size differences, not the call-
count/overhead difference that's the actual point of RSQR. Confirmed
via the run: A calls == B calls at every n_steps value, and the
'speedup' bounced around with no trend, which is what you'd expect
from two nearly-identical workloads plus GPU timing noise.

Corrected model: continuous re-rotation re-rotates the ENTIRE current
window on **every single step** (not just eviction steps) -- the
strict StreamingLLM reading, since a sliding window's positions are
conceptually kept fresh at every step, not just when eviction fires.
Eviction still happens at `evict_every` cadence (removing the oldest
tokens), but the full-window rotation call now fires every step
regardless. This makes n_rotation_calls scale with `n_steps` for Arm
A, vs `n_steps // evict_every` for Arm B -- a real, structural
difference instead of an artifact of matched gating.


In [ ]:
@torch.no_grad()
def run_arm_a_continuous(cfg: BenchConfig, initial_k):
    """Simulates STRICT continuous per-step re-rotation. Returns
    (n_rotation_calls, total_tokens_rotated). Window grows by 1 token per
    step; every `evict_every` steps the oldest evict_every tokens are
    evicted. Critically -- and unlike v1 of this cell -- the full
    surviving window is re-rotated to its current relative positions on
    EVERY step, not just eviction steps. This is what "continuous"
    actually means: positions are kept fresh at every step, so
    n_rotation_calls scales with n_steps here, not with n_steps //
    evict_every like Arm B below.
    """
    k = initial_k.clone()
    n_rotation_calls = 0
    total_tokens_rotated = 0
    cumulative_shift = 0  # tracks how far positions have drifted since last real eviction

    for step in range(1, cfg.n_steps + 1):
        new_tok = torch.randn(BATCH_SIZE, N_KV_HEADS, 1, HEAD_DIM, device=device)
        k = torch.cat([k, new_tok], dim=2)

        if step % cfg.evict_every == 0 and k.shape[2] > cfg.evict_every:
            evict_n = cfg.evict_every
            k = k[:, :, evict_n:, :]
            cumulative_shift += evict_n

        # Full-window re-rotation EVERY step, regardless of whether this
        # step evicted anything -- this is the "continuous" cost model.
        seq_len = k.shape[2]
        if cumulative_shift > 0:
            shift_positions = torch.full((seq_len,), float(-cumulative_shift), device=device)
            k = apply_rope(k, shift_positions, FREQS)
            n_rotation_calls += 1
            total_tokens_rotated += seq_len

    return n_rotation_calls, total_tokens_rotated


## Arm B -- RSQR (batched, eviction-boundary only)

Only newly-flagged survivor tokens get rotated, only at eviction
boundaries, in a single batched call per eviction event. The rest of
the window is never touched -- that's the entire mechanism being
tested.


In [ ]:
@torch.no_grad()
def run_arm_b_rsqr(cfg: BenchConfig, initial_k):
    """Simulates RSQR's batched, eviction-boundary-only rotation. Returns
    (n_rotation_calls, total_tokens_rotated). Survivors accumulate in a
    separate raw-shadow store (never rotated until needed); at each
    eviction boundary, only the newly-flagged `survivor_delta` tokens for
    this cycle get rotated, in ONE batched call -- not the whole window.
    """
    k = initial_k.clone()
    survivor_raw = torch.zeros(BATCH_SIZE, N_KV_HEADS, 0, HEAD_DIM, device=device)
    n_rotation_calls = 0
    total_tokens_rotated = 0

    for step in range(1, cfg.n_steps + 1):
        new_tok = torch.randn(BATCH_SIZE, N_KV_HEADS, 1, HEAD_DIM, device=device)
        k = torch.cat([k, new_tok], dim=2)

        if step % cfg.evict_every == 0 and k.shape[2] > cfg.evict_every:
            evict_n = cfg.evict_every
            evicted = k[:, :, :evict_n, :]
            k = k[:, :, evict_n:, :]

            # Flag survivor_delta of the evicted tokens as permanent
            # survivors (raw, pre-rotation) -- matches Phase 3's
            # every-Delta-th-token flagging rule.
            n_flag = min(cfg.survivor_delta, evicted.shape[2])
            survivor_raw = torch.cat([survivor_raw, evicted[:, :, :n_flag, :]], dim=2)

            # Single batched rotation call over ALL current survivors
            # (rotate-from-raw to their current logical position) -- not
            # the whole window, and not a running/compounding rotation.
            n_survivors = survivor_raw.shape[2]
            if n_survivors > 0:
                target_positions = torch.arange(n_survivors, dtype=torch.float32, device=device)
                _ = apply_rope(survivor_raw, target_positions, FREQS)
                n_rotation_calls += 1
                total_tokens_rotated += n_survivors

    return n_rotation_calls, total_tokens_rotated


## Timing harness

**This is the part most likely to lie to you if done wrong.** Rules
followed here:
- `torch.cuda.Event(enable_timing=True)` pairs, not `time.time()` --
  wall-clock timers around GPU code don't account for async kernel
  execution and will report near-zero times that mean nothing.
- `torch.cuda.synchronize()` before starting AND after stopping each
  timed run -- without this you measure kernel *launch* (near-
  instant), not kernel *completion*.
- Warmup phase (discarded) before every timed measurement, to avoid
  counting one-time CUDA context/kernel-compile cost.
- At least 20 repeats per config, reporting mean/median/std -- GPU
  timing has real run-to-run variance; a single number isn't
  trustworthy.


In [ ]:
def time_arm(run_fn, cfg: BenchConfig, initial_k):
    """Runs `run_fn` cfg.warmup_trials times (discarded), then
    cfg.n_trials times (timed). Returns dict with timing stats plus the
    call-count/token-count metrics from the LAST timed run (these are
    deterministic given cfg, so any single run's counts are
    representative)."""
    # Warmup -- discard results, just let CUDA context/kernels settle.
    for _ in range(cfg.warmup_trials):
        run_fn(cfg, initial_k)
    if device == "cuda":
        torch.cuda.synchronize()

    times_ms = []
    n_calls = n_tokens = None

    for _ in range(cfg.n_trials):
        if device == "cuda":
            torch.cuda.synchronize()
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            n_calls, n_tokens = run_fn(cfg, initial_k)
            end.record()
            torch.cuda.synchronize()
            times_ms.append(start.elapsed_time(end))
        else:
            t0 = time.perf_counter()
            n_calls, n_tokens = run_fn(cfg, initial_k)
            t1 = time.perf_counter()
            times_ms.append((t1 - t0) * 1000.0)

    return {
        "mean_ms": statistics.mean(times_ms),
        "median_ms": statistics.median(times_ms),
        "std_ms": statistics.stdev(times_ms) if len(times_ms) > 1 else 0.0,
        "n_rotation_calls": n_calls,
        "total_tokens_rotated": n_tokens,
    }


## Sweep

Default sweep matches the CLI spec: `n_steps in [100, 500, 1000, 5000]`.
Change `N_STEPS_SWEEP` below to adjust.


In [ ]:
N_STEPS_SWEEP = [100, 500, 1000, 5000]
EVICT_EVERY = 8
SURVIVOR_DELTA = 8
N_TRIALS = 20
OUTPUT_CSV = "latency_results.csv"

results = []

for n_steps in N_STEPS_SWEEP:
    cfg = BenchConfig(n_steps=n_steps, evict_every=EVICT_EVERY,
                       survivor_delta=SURVIVOR_DELTA, n_trials=N_TRIALS)

    # Same starting cache state for both arms (cloned per call inside
    # each run_fn) so the comparison is fair -- neither arm gets a head
    # start from a different initial window size.
    initial_seq_len = 24  # matches Phase 3's window_size
    initial_k = torch.randn(BATCH_SIZE, N_KV_HEADS, initial_seq_len, HEAD_DIM, device=device)

    stats_a = time_arm(run_arm_a_continuous, cfg, initial_k)
    stats_b = time_arm(run_arm_b_rsqr, cfg, initial_k)

    speedup = stats_a["mean_ms"] / stats_b["mean_ms"] if stats_b["mean_ms"] > 0 else float("inf")

    row = {
        "n_steps": n_steps,
        "arm_a_mean_ms": stats_a["mean_ms"], "arm_a_median_ms": stats_a["median_ms"], "arm_a_std_ms": stats_a["std_ms"],
        "arm_b_mean_ms": stats_b["mean_ms"], "arm_b_median_ms": stats_b["median_ms"], "arm_b_std_ms": stats_b["std_ms"],
        "speedup_a_over_b": speedup,
        "arm_a_rotation_calls": stats_a["n_rotation_calls"], "arm_b_rotation_calls": stats_b["n_rotation_calls"],
        "arm_a_tokens_rotated": stats_a["total_tokens_rotated"], "arm_b_tokens_rotated": stats_b["total_tokens_rotated"],
    }
    results.append(row)

    print(f"n_steps={n_steps:5d} | A: {stats_a['mean_ms']:8.3f}ms (calls={stats_a['n_rotation_calls']:4d}, tokens={stats_a['total_tokens_rotated']:6d}) | "
          f"B: {stats_b['mean_ms']:8.3f}ms (calls={stats_b['n_rotation_calls']:4d}, tokens={stats_b['total_tokens_rotated']:6d}) | "
          f"speedup={speedup:.2f}x")

with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)
print(f"\nSaved results to {OUTPUT_CSV}")


n_steps=  100 | A:   18.260ms (calls=  93, tokens=  2550) | B:    4.918ms (calls=  12, tokens=   624) | speedup=3.71x
n_steps=  500 | A:   96.594ms (calls= 493, tokens= 13550) | B:   24.875ms (calls=  62, tokens= 15624) | speedup=3.88x
n_steps= 1000 | A:  225.015ms (calls= 993, tokens= 27304) | B:   54.257ms (calls= 125, tokens= 63000) | speedup=4.15x
n_steps= 5000 | A: 1244.572ms (calls=4993, tokens=137304) | B:  271.469ms (calls= 625, tokens=1565000) | speedup=4.58x

Saved results to latency_results.csv


## §5#2 — Bump severity vs. window size/Delta

Untouched question until now. The existing sweep above varies
`n_steps` only, with `evict_every=8` and `survivor_delta=8` fixed
throughout -- it never tests whether RSQR's speedup advantage holds,
grows, or shrinks as eviction *severity* (how many tokens get
evicted/flagged per boundary event) changes.

Two axes tested separately below, holding `n_steps` fixed so results
are comparable to each other:

1. **`evict_every` sweep** (bump frequency/severity) -- smaller
   `evict_every` means more frequent, smaller eviction events; larger
   means rarer, bigger ones. `survivor_delta` is kept equal to
   `evict_every` in this pass (same ratio as the original Phase 3/4
   default) so the flagging RATE stays constant relative to eviction
   size -- isolates the effect of bump frequency alone.
2. **`survivor_delta` sweep**, `evict_every` held fixed at 8 -- isolates
   how many tokens get flagged as permanent survivors per eviction,
   independent of how often evictions happen. This is the more direct
   read on "how many survivors accumulate over time" and connects
   directly to the survivor-count-bounding gap found in Phase 3
   (n_cycles=64 collapse) -- if RSQR's speedup shrinks or reverses as
   `survivor_delta` grows, that's a second, independent reason (beyond
   the accuracy collapse already found) to cap survivor count.

In [ ]:
N_STEPS_FIXED = 2000  # fixed stream length for both severity sweeps below

# ---- Sweep 1: evict_every (bump frequency/severity), survivor_delta tied to it ----
EVICT_EVERY_SWEEP = [2, 4, 8, 16, 32, 64]

severity_results = []
for evict_every in EVICT_EVERY_SWEEP:
    cfg = BenchConfig(n_steps=N_STEPS_FIXED, evict_every=evict_every,
                       survivor_delta=evict_every, n_trials=N_TRIALS)
    initial_k = torch.randn(BATCH_SIZE, N_KV_HEADS, 0, HEAD_DIM, device=device)

    stats_a = time_arm(run_arm_a_continuous, cfg, initial_k)
    stats_b = time_arm(run_arm_b_rsqr, cfg, initial_k)

    speedup = stats_a["mean_ms"] / stats_b["mean_ms"] if stats_b["mean_ms"] > 0 else float("nan")
    row = {
        "evict_every": evict_every,
        "survivor_delta": evict_every,
        "arm_a_mean_ms": stats_a["mean_ms"],
        "arm_b_mean_ms": stats_b["mean_ms"],
        "speedup_b_over_a": speedup,
        "arm_a_rotation_calls": stats_a["n_rotation_calls"],
        "arm_b_rotation_calls": stats_b["n_rotation_calls"],
    }
    severity_results.append(row)
    print(f"evict_every={evict_every:3d}  A={stats_a['mean_ms']:.3f}ms  "
          f"B={stats_b['mean_ms']:.3f}ms  speedup={speedup:.2f}x  "
          f"calls A/B={stats_a['n_rotation_calls']}/{stats_b['n_rotation_calls']}")

print()
print("=" * 70)
print("Sweep 1 summary -- does speedup hold as bump frequency/severity changes?")
print("=" * 70)
speedups_1 = [r["speedup_b_over_a"] for r in severity_results]
print(f"  speedup range: {min(speedups_1):.2f}x - {max(speedups_1):.2f}x")
print(f"  monotonic trend: ", end="")
diffs = [speedups_1[i+1] - speedups_1[i] for i in range(len(speedups_1)-1)]
if all(d >= -0.05 for d in diffs):
    print("non-decreasing (speedup holds or grows as evict_every increases)")
elif all(d <= 0.05 for d in diffs):
    print("non-increasing (speedup shrinks as evict_every increases)")
else:
    print("NOT monotonic -- speedup varies non-uniformly, inspect table above")


In [ ]:
# ---- Sweep 2: survivor_delta alone, evict_every fixed at 8 ----
SURVIVOR_DELTA_SWEEP = [2, 4, 8, 16, 32, 64]

delta_results = []
for survivor_delta in SURVIVOR_DELTA_SWEEP:
    cfg = BenchConfig(n_steps=N_STEPS_FIXED, evict_every=8,
                       survivor_delta=survivor_delta, n_trials=N_TRIALS)
    initial_k = torch.randn(BATCH_SIZE, N_KV_HEADS, 0, HEAD_DIM, device=device)

    stats_a = time_arm(run_arm_a_continuous, cfg, initial_k)
    stats_b = time_arm(run_arm_b_rsqr, cfg, initial_k)

    speedup = stats_a["mean_ms"] / stats_b["mean_ms"] if stats_b["mean_ms"] > 0 else float("nan")
    row = {
        "survivor_delta": survivor_delta,
        "evict_every": 8,
        "arm_a_mean_ms": stats_a["mean_ms"],
        "arm_b_mean_ms": stats_b["mean_ms"],
        "speedup_b_over_a": speedup,
        "arm_b_total_tokens_rotated": stats_b["total_tokens_rotated"],
    }
    delta_results.append(row)
    print(f"survivor_delta={survivor_delta:3d}  A={stats_a['mean_ms']:.3f}ms  "
          f"B={stats_b['mean_ms']:.3f}ms  speedup={speedup:.2f}x  "
          f"B_tokens_rotated={stats_b['total_tokens_rotated']}")

print()
print("=" * 70)
print("Sweep 2 summary -- does more survivor accumulation erode RSQR's speedup?")
print("=" * 70)
speedups_2 = [r["speedup_b_over_a"] for r in delta_results]
print(f"  speedup range: {min(speedups_2):.2f}x - {max(speedups_2):.2f}x")
print(f"  NOTE: this sweep uses the SAME uncapped survivor accumulation as the")
print(f"  rest of this notebook -- survivor count grows without bound as")
print(f"  n_steps/n_cycles increase (same open issue as Phase 3's accuracy")
print(f"  collapse at n_cycles=64). If speedup degrades noticeably as")
print(f"  survivor_delta grows, that's a SECOND, latency-side reason (beyond")
print(f"  the accuracy collapse already found) to implement a survivor cap --")
print(f"  not fixed here, this cell only measures the effect.")

import csv
with open("severity_sweep_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(severity_results[0].keys()))
    writer.writeheader()
    writer.writerows(severity_results)
with open("survivor_delta_sweep_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(delta_results[0].keys()))
    writer.writeheader()
    writer.writerows(delta_results)
print("\nResults saved to severity_sweep_results.csv and survivor_delta_sweep_results.csv")


## Plain-language summary

The actual answer to §5#1 -- read this, not just the raw table.


In [ ]:
print("=" * 70)
print("SANITY CHECK -- do the two arms actually differ in call count?")
print("=" * 70)
for r in results:
    if r["arm_a_rotation_calls"] == r["arm_b_rotation_calls"]:
        print(f"  n_steps={r['n_steps']:5d}: WARNING -- A and B have IDENTICAL "
              f"rotation call counts ({r['arm_a_rotation_calls']}). This means "
              f"the benchmark isn't measuring the mechanism difference RSQR is "
              f"supposed to provide -- check run_arm_a_continuous and "
              f"run_arm_b_rsqr before trusting the speedup numbers below.")
    else:
        ratio = r["arm_a_rotation_calls"] / r["arm_b_rotation_calls"] if r["arm_b_rotation_calls"] else float("inf")
        print(f"  n_steps={r['n_steps']:5d}: OK -- A made {r['arm_a_rotation_calls']} calls, "
              f"B made {r['arm_b_rotation_calls']} calls ({ratio:.1f}x more calls in A)")

print()
print("=" * 70)
print("SUMMARY -- does the RSQR-vs-continuous speedup grow, shrink, or")
print("stay flat as stream length (n_steps) increases?")
print("=" * 70)

speedups = [r["speedup_a_over_b"] for r in results]
for r in results:
    faster = "RSQR (Arm B)" if r["speedup_a_over_b"] > 1.0 else "Continuous (Arm A)"
    print(f"  n_steps={r['n_steps']:5d}: {faster} faster, "
          f"{r['speedup_a_over_b']:.2f}x  "
          f"(A calls={r['arm_a_rotation_calls']}, B calls={r['arm_b_rotation_calls']})")

print()
if len(speedups) >= 2:
    if speedups[-1] > speedups[0] * 1.1:
        trend = "GROWS as n_steps increases -- RSQR's advantage compounds with stream length."
    elif speedups[-1] < speedups[0] * 0.9:
        trend = "SHRINKS as n_steps increases -- worth checking whether call overhead is being amortized differently than expected."
    else:
        trend = "stays roughly FLAT across n_steps -- the per-call overhead difference dominates, independent of stream length."
    print(f"Trend: speedup {trend}")

print()
print("Reminder from the progress log: RAP (arXiv 2602.02599) found RoPE")
print("itself is <1% of inference latency unfused -- so if RSQR wins here,")
print("the win should track ROTATION CALL COUNT (kernel-launch overhead),")
print("not total tokens rotated. Check the calls columns above against")
print("the tokens-rotated columns to confirm this is the actual mechanism")
print("before citing a speedup number.")


SANITY CHECK -- do the two arms actually differ in call count?
  n_steps=  100: OK -- A made 93 calls, B made 12 calls (7.8x more calls in A)
  n_steps=  500: OK -- A made 493 calls, B made 62 calls (8.0x more calls in A)
  n_steps= 1000: OK -- A made 993 calls, B made 125 calls (7.9x more calls in A)
  n_steps= 5000: OK -- A made 4993 calls, B made 625 calls (8.0x more calls in A)

SUMMARY -- does the RSQR-vs-continuous speedup grow, shrink, or
stay flat as stream length (n_steps) increases?
  n_steps=  100: RSQR (Arm B) faster, 3.71x  (A calls=93, B calls=12)
  n_steps=  500: RSQR (Arm B) faster, 3.88x  (A calls=493, B calls=62)
  n_steps= 1000: RSQR (Arm B) faster, 4.15x  (A calls=993, B calls=125)
  n_steps= 5000: RSQR (Arm B) faster, 4.58x  (A calls=4993, B calls=625)

Trend: speedup GROWS as n_steps increases -- RSQR's advantage compounds with stream length.

Reminder from the progress log: RAP (arXiv 2602.02599) found RoPE
itself is <1% of inference latency unfused -- so if RSQR